In [ ]:
from pyspark.sql import SparkSession
from openai import OpenAI
import json
import os
from typing import List, Optional, Dict, Any, Literal, TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END
#close existing spark session  
if "spark" in locals():
  spark.stop()
  print("Existing Spark session stopped")

spark = SparkSession.builder \
    .appName("BT4221") \
    .master("local[2]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.3") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark running:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/07 19:08:19 WARN Utils: Your hostname, Bryans-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 172.31.212.199 instead (on interface en0)
26/04/07 19:08:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 19:08:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/07 19:08:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark running: 4.1.1


## load datasets

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
ORIGINAL_CSV_PATH = PROJECT_ROOT / "US_Accidents_March23.csv"
CLEANED_PARQUET_DIR = PROJECT_ROOT / "cleaned_parquet"

if not ORIGINAL_CSV_PATH.exists():
    raise FileNotFoundError(f"Original CSV not found at: {ORIGINAL_CSV_PATH}")

if not CLEANED_PARQUET_DIR.exists():
    raise FileNotFoundError(
        f"Cleaned parquet folder not found at: {CLEANED_PARQUET_DIR}\n"
        "Update CLEANED_PARQUET_DIR to your actual folder."
    )


print("Project root:", PROJECT_ROOT)
print("Original CSV:", ORIGINAL_CSV_PATH)
print("Cleaned parquet dir:", CLEANED_PARQUET_DIR)

raw_df = spark.read.csv(str(ORIGINAL_CSV_PATH), header=True, inferSchema=True)
cleaned_df = spark.read.parquet(str(CLEANED_PARQUET_DIR))

print("Raw rows / cols:", raw_df.count(), len(raw_df.columns))
print("Cleaned rows / cols:", cleaned_df.count(), len(cleaned_df.columns))

Project root: /Users/bryan/Documents/Y2S2/BT4221/Tutorials/work
Original CSV: /Users/bryan/Documents/Y2S2/BT4221/Tutorials/work/US_Accidents_March23.csv
Cleaned parquet dir: /Users/bryan/Documents/Y2S2/BT4221/Tutorials/work/cleaned_parquet


Raw rows / cols: 7728394 46
Cleaned rows / cols: 5270673 50


In [ ]:
cleaned_df.show()

+------------+----------------+--------+-------------------+-------------------+-----------+-------------------+--------------------+-----------------+-------------+--------+-----+----------+-----------+--------------+-----------+------------+--------------+--------------+---------------+-----------------+--------------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+--------------+--------------+-----------------+---------------------+---------------+----------------------+---------------+---------------------+-------------+--------------+--------------------+-------------+-------------------+----------------+------------------+-------------------+
|Airport_Code|Start_Time_Month|Severity|         Start_Time|           End_Time|  Start_Lat|          Start_Lng|        Distance(mi)|           Street|         City|  County|State|   Zipcode|   Timezone|Temperature(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Directio

In [ ]:
def create_secure_openai_client():
    """
    Create OpenAI client with secure API key handling.

    This function:
    1. Looks for OPENAI_API_KEY in environment variables
    2. Tests the connection with a simple API call
    3. Returns the client or None if setup fails
    """
    try:
        from dotenv import load_dotenv
        load_dotenv()  # Load .env file if it exists
    except ImportError:
        pass  # python-dotenv not installed, that's okay

    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        print("No OpenAI API key found.")
        print("Set environment variable: OPENAI_API_KEY=your_key")
        print("Or create .env file with: OPENAI_API_KEY=your_key")
        return None

    try:
        client = OpenAI(api_key=api_key)
        # Test connection with a simple API call
        models = client.models.list()
        print("OpenAI client created and tested successfully")
        return client
    except Exception as e:
        print(f"OpenAI client creation failed: {e}")
        print("Check your API key and internet connection")
        return None

# Initialize the client
client = create_secure_openai_client()

OpenAI client created and tested successfully


## Define Skills

### skill_bin_numeric

In [ ]:
SKILL_BIN_NUMERIC="""
---
name: skill_bin_numeric.md
description: Discretise a continuous numeric column into ordered integer bins using known thresholds
mode: orgnisational
---

# skill_bin_numeric.md

## Purpose

Discretise a continuous numeric column into ordered integer bins.
Useful when a numeric feature has known meaningful thresholds that
a tree model might not discover efficiently on its own.

## When the agent should call this skill

Apply this skill to continuous numeric columns where domain knowledge dictates meaningful thresholds (e.g., freezing temperatures, visibility limits). You are expected to apply this to highly sensitive environmental factors like Temperature(F) and Visibility(mi).

## Recommended candidates for this dataset

| Column          | Suggested cuts   | Reasoning                        |
| --------------- | ---------------- | -------------------------------- |
| Visibility(mi)  | [0.25, 1.0, 5.0] | Near-zero, poor, moderate, clear |
| Temperature(F)  | [32, 50, 80]     | Freezing, cold, mild, hot        |
| Wind_Speed(mph) | [15, 30]         | Calm, breezy, high wind          |

## Available strategies

| Strategy    | When to use                           | Required params   |
| ----------- | ------------------------------------- | ----------------- |
| equal_width | No strong domain knowledge about cuts | n_bins            |
| custom_cuts | Meaningful thresholds known from EDA  | cuts: list[float] |

## Input

- df: PySpark DataFrame
- column: str
- strategy: "equal_width" or "custom_cuts"
- n_bins: int (equal_width only)
- cuts: list[float] (custom_cuts only)

## Output

DataFrame with new {column}_bin integer column added.
Original column is retained, agent decides whether to keep or drop it
by including/excluding both in assembler_cols.

## Constraints

- Do not bin is_weekend, is_rush_hour, hour these are already
  low-cardinality integers, binning adds no value
- Output column name for assembler_cols: {column}_bin

"""

with open("SKILLS/skill_bin_numeric.md", "w") as f:
    f.write(SKILL_BIN_NUMERIC.strip())
print("Written: SKILLS/skill_bin_numeric.md")

### skill_compute_interaction_features

In [ ]:
SKILL_COMPUTE_INTERACTION_FEATURES="""
---
name: skill_compute_interactions_features
description: Create new features by combining two existing columns to capture interactions effects
mode: organisational
---

# skill_compute_interaction_features.md

## Purpose

Create new features by combining two existing columns. Captures
compound effects that individual features may not express alone.

## When the agent should call this skill

Only when domain reasoning supports the interaction. Each interaction
adds one column to the feature vector.

## Recommended candidates for this dataset

(Note: These are just examples. You are expected to discover your own novel interaction features based on the exact columns provided in the runtime data profile)
| Interaction | Op | Reasoning |
| ------------------------------- | -------- | -------------------------------------------------- |
| hour × is_weekend | multiply | Weekend nights are distinct risk profile |
| Temperature(F) x Humidty(%) | multiply | High temp + high humidity compounds weather stress |
| Distance(mi) x Duration_Minutes | ratio | Captures the speed/impact spread of the accident |

## Available operations

| Op       | Formula                | Use when                         |
| -------- | ---------------------- | -------------------------------- |
| multiply | col_a * col_b         | Both columns contribute jointly  |
| add      | col_a + col_b          | Additive combination makes sense |
| ratio    | col_a / (col_b + 1e-6) | One column normalises the other  |

## Input

- df: PySpark DataFrame
- col_a: str (MUST exist in the Dataframe)
- col_b: str (MUST exist in the Dataframe)
- op: "multiply", "add", or "ratio"

## Output

DataFrame with new column named {col_a}_{op}_{col_b}.

## Constraints

- Both input columns must already exist in the DataFrame at time of call
- NEVER invent columns that are not in the data profile
- Output column name for assembler cols: {col_a}_{op}_{col_b} where op is one of "multiply", "add", "ratio"
- Avoid creating interactions blindly, each interaction adds dimensionality

"""

with open("SKILLS/skill_bin_numeric.md", "w") as f:
    f.write(SKILL_COMPUTE_INTERACTION_FEATURES.strip())
print("Written: SKILLS/skill_compute_interaction_features.md")

### skill_drop_columns

In [ ]:
SKILL_DROP_COLUMNS="""
---
name: skill_drop_columns
description: Remove columns from the DataFrame after encoding is complete to keep the feature space clean
mode: organisational
---

# skill_drop_columns.md

## Purpose

Remove columns from the DataFrame after encoding is complete.
Keeps the feature space clean before VectorAssembler runs.

## When the agent should call this skill

Always after encoding, original string columns and Start_Time
must be removed. They cannot coexist with their encoded counterparts
in the assembler input without causing duplication.

## What to include in drop_after_encoding

Always drop:

- Start_Time (replaced by extracted time features)
- Original string columns after string_index_ohe or string_index_only
  e.g. if State → State_ohe, drop State and State_idx
- Any column explicitly marked action="drop" in column_encodings

Do NOT drop:

- Passthrough columns (already numeric, include directly)
- Boolean columns (is_rain, is_fog etc. — include in assembler_cols)
- The target column Severity_Binary

## Input

- df: PySpark DataFrame
- columns: list[str] — columns to remove

## Output

DataFrame with specified columns removed.

## Constraints

- The executor will only drop columns that exist — safe to include
  columns that may or may not be present

"""

with open("SKILLS/skill_drop_columns.md", "w") as f:
    f.write(SKILL_COMPUTE_INTERACTION_FEATURES.strip())
print("Written: SKILLS/skill_drop_columns.md")

### skill_encode_categorical

In [ ]:
SKILL_ENCODE_CATEGORICAL="""
---
name: skill_encode_categorical
description: convert string categorical columns to numeric form compatible with PySpark MLlib. MLlib cannnot consume raw string columns
mode: orgnisational
---

# skill_encode_categorical.md

## Purpose

Convert string categorical columns to numeric form compatible with
PySpark MLlib. MLlib cannot consume raw string columns.

## When the agent should call this skill

For every string column in the cleaned data profile that is not being dropped.

## Available methods

| Method            | When to use                       | Output column     |
| ----------------- | --------------------------------- | ----------------- |
| string_index_ohe  | n_unique <= 20                    | {col}_ohe        |
| string_index_only | n_unique > 20                     | {col}_idx        |
| passthrough       | column is already numeric/boolean | {col} (unchanged) |
| drop              | column has no predictive value    | column removed    |

## Detail

**string_index_ohe**: Applies StringIndexer then OneHotEncoder.
Produces a sparse binary vector. Use when cardinality is low enough
that the resulting dimensions won't explode the feature vector.
Threshold: n_unique <= 20.

**string_index_only**: Applies StringIndexer only. Assigns an integer
index to each category. Use for high-cardinality columns (e.g. City
with 9000+ unique values) where OHE would produce thousands of dimensions.

**passthrough**: Column is already in a numeric form MLlib can consume
(double, int, boolean). No transformation needed. Use for all boolean
columns produced by the weather condition expansion (is_rain, is_fog etc.)
and for columns already binary-encoded.

**drop**: Column carries no signal useful for severity prediction.
Use for free-text columns (Description) and columns made redundant
by derived features.

## Input

- df: PySpark DataFrame
- column: str — column name
- method: one of the four methods above

## Output

DataFrame with encoded column added. Original string column is NOT
dropped here, add it to drop_after_encoding in FeatureEngDecision.

## Assembler col naming

After encoding, use the output column name in assembler_cols:

- string_index_ohe → {col}_ohe
- string_index_only → {col}_idx
- passthrough → {col}

"""

with open("SKILLS/skill_encode_categorical.md", "w") as f:
    f.write(SKILL_COMPUTE_INTERACTION_FEATURES.strip())
print("Written: SKILLS/skill_encode_categorical.md")

### skill_extract_time_features

In [ ]:
SKILL_EXTRACT_TIME_FEATURES="""
---
name: skill_extract_time_features
description: Decompose the start_time timestamp column into numeric temporal features
mode: organisational
---

# skill_extract_time_features.md

## Purpose

Decompose the Start_Time timestamp column into numeric temporal features
usable by PySpark MLlib. Raw timestamps cannot be consumed by tree models
or logistic regression directly.

## When the agent should call this skill

Always. Start_Time must be decomposed before assembly.
Use when `is_rush_hour` is not yet present

## Available features

The agent selects any subset of the following:

| Feature      | PySpark derivation           | Reasoning               |
| ------------ | ---------------------------- | ----------------------- |
| is_rush_hour | hour in [7-9] or [17-19] → 1 | Peak congestion periods |

## Input

- df: PySpark DataFrame
- features_to_extract: list of feature names from the table above

## Output

DataFrame with new integer columns added.

"""

with open("SKILLS/skill_extract_time_features.md", "w") as f:
    f.write(SKILL_COMPUTE_INTERACTION_FEATURES.strip())
print("Written: SKILLS/skill_extract_time_features.md")

### skill_scale_numeric

In [ ]:
SKILL_SCALE_NUMERIC="""
---
name: skill_scale_numeric
description: Scale continuous numeric columns to a common range for distance based or gradient based models
mode: organisational
---

# skill_scale_numeric.md

## Purpose

Scale continuous numeric columns to a common range. Required for
distance-based or gradient-based models.

## When the agent should call this skill

ALWAYS call this skill.
Since Logistic Regression is being used as the baseline model, continuous features MUST be scaled for the model to converge properly and provide valid coefficients. While advanced models like RandomForest and GBT are scale-invariant, feeding them scaled data does not harm their performance. Applying scaling here ensures the final dataset is universally compatible across all our planned model pipelines.

## Available strategies

| Strategy | Formula                 | When to use                        |
| -------- | ----------------------- | ---------------------------------- |
| standard | (x - mean) / stddev     | Default for logistic regression    |
| minmax   | (x - min) / (max - min) | When bounded [0,1] range is needed |

## Columns to scale (if scaling is applied)

Apply to all continuous numeric columns together:
Temperature(F), Humidity(%), Pressure(in), Visibility(mi), Wind_Speed(mph)

## Do NOT scale

- Binary columns (is_weekend, is_rush_hour, Sunrise_Sunset)
- Boolean columns (is_rain, is_fog etc.)
- One-hot encoded columns ({col}_ohe)
- Integer index columns ({col}_idx)

## Input

- df: PySpark DataFrame
- columns: list[str] — numeric columns to scale
- strategy: "standard" or "minmax"

## Output

DataFrame with scaled columns replacing originals (same column names).

"""

with open("SKILLS/skill_scale_numeric.md", "w") as f:
    f.write(SKILL_COMPUTE_INTERACTION_FEATURES.strip())
print("Written: SKILLS/skill_scale_numeric.md")

### skill_semantic_boolean_expansion

In [ ]:
SKILL_SEMANTIC_BOOELAN_EXPANSION="""
---
name: skill_semantic_boolean_expansion
description: Expand high cardinality text columns into multiple independent binary indicators
mode: organisational
---

# skill_semantic_boolean_expansion.md

## Purpose

Convert compound, descriptive text columns (eg. Weather_Condition or Street) into multiple independent binary indicators (eg. is_rain, is_highway).

## When the agent should call this skill

When a categorical string column contains overlapping, compound information or has extreme cardinality (eg. >100000 unique values) that is better represented as boolean flag rather than a massive one hot encoded vector or integer index

## Reccommended candiddates for this dataset

- Note these are just examples. You are expected to use your domain knowledge to write regex patterns that capture broad categories\*

**Weather_Condition Examples**
| Semantic Target | Example Regex Pattern | Expected Target Column |
| --- | --- | --- |
| Rain/Storms | "rain\|storm\|drizzle\|shower\|thunder" | is_rain |
| Snow/Ice | "snow\|sleet\|ice\|pellets\|squall" | is_snow |
| Fog/Visibility | "fog\|haze\|mist" | is_fog |

**Street Examples:**
| Semantic Target | Example Regex Pattern | Expected Target Column |
| --- | --- | --- |
| Interstate / Highway | "i-\|hwy\|highway\|expressway\|freeway" | is_highway |
| County / State Roads | "cr-\|sr-\|county\|state" | is_county_road |
| Local Street / Avenue | "st\|ave\|blvd\|rd\|road\|drive\|dr" | is_local_street |

## Input

- df: PySpark DataFrame
- source_column: str (The compound text column, e.g., "Weather_Condition" or "Street")
- target_column: str (The new boolean column name, e.g., "is_highway")
- regex_pattern: str (The regex pattern to search for, in lowercase)

## Output

DataFrame with a new integer column (1 if regex matches, 0 otherwise).

## Constraints

- The executor will automatically lowercase the source column before applying your regex. Write all your regex patterns in strictly lowercase.
- Use simple OR pipes (`|`) for your regex.
- You must add the newly created `target_column` names to `assembler_cols`.
- You must add the original `source_column` to `drop_after_encoding`.
- CRITICAL: You MUST create MULTIPLE (at least 5-6) SemanticExpansionDecisions for EACH high-cardinality column, not just one! For instance, if expanding 'Weather_Condition', create separate decisions for 'is_rain', 'is_snow', 'is_fog', etc.

"""

with open("SKILLS/skill_semantic_boolean_expansion.md", "w") as f:
    f.write(SKILL_COMPUTE_INTERACTION_FEATURES.strip())
print("Written: SKILLS/skill_semantic_boolean_expansion.md")

## Define pydantic schemas inline with our various skills

In [ ]:
class TimeFeatureDecision(BaseModel):
    features_to_extract: List[Literal['is_rush_hour', ]]
    rationale: str

class ColumnEncodingDecision(BaseModel):
    column: str
    action: Literal['string_index_ohe', 'string_index_only', 'passthrough', 'drop']
    rationale: str

class BinSpec(BaseModel):
    column: str
    strategy: Literal["equal_width", "custom_cuts"]
    n_bins: Optional[int]=None
    cuts: Optional[List[float]] =None
    rationale: str

class InteractionSpecs(BaseModel):
    col_a: str
    col_b: str
    op: Literal["multiply","add", "ratio"]
    rationale: str

class ScalingDecisions(BaseModel):
    column: str
    action: Literal['scale', 'skip']
    rationale: str

class SemanticExpansionDecision(BaseModel):
    source_column: Literal["Weather_Condition", "Street"]
    target_column: str
    regex_pattern: str
    rationale: str
    support_estimate_pct: Optional[float] = None

class FeatureEngDecision(BaseModel):
    time_features: TimeFeatureDecision
    semantic_expansions: List[SemanticExpansionDecision] = Field(default_factory=list)
    column_encodings: List[ColumnEncodingDecision]
    bins: Optional[List[BinSpec]] = None
    interactions: Optional[List[InteractionSpecs]] = None

    apply_scaling: bool
    scaling_strategy: Optional[Literal["standard", "minmax"]] = None
    scaling_decisions: Optional[List[ScalingDecisions]] = None
    scaling_rationale: Optional[str] = None

    drop_after_encoding: List[str]
    assembler_cols: List[str]
    overall_rationale: str

# schemas for subagent ouputs making use of the above defined schemas

# for semantic agent
class SemanticDecisionList(BaseModel):
    semantic_expansions: List[SemanticExpansionDecision] = Field(default_factory=list)
    overall_rationale: str

# for numeric agent
class NumericDecisionList(BaseModel):
    time_features: Optional[TimeFeatureDecision] = None
    bins: Optional[List[BinSpec]] = Field(default_factory=list)
    interactions: Optional[List[InteractionSpecs]] = Field(default_factory=list)
    apply_scaling: bool
    scaling_strategy: Optional[Literal["standard", "minmax"]] = None
    scaling_decisions: Optional[List[ScalingDecisions]] = Field(default_factory=list)
    scaling_rationale: Optional[str] = None
    overall_rationale: str

# for categorical agent
class CategoricalDecisionList(BaseModel):
    column_encodings: List[ColumnEncodingDecision]
    drop_after_encoding: List[str]
    overall_rationale: str


## Agent state for langgraph, needs to be modified to be consistent with earlier pipeline stages eventually

In [ ]:
class AgentState(TypedDict):
    # --- Phase 1: Manual Cleaning ---
    raw_df_path: str
    cleaned_df_path: str
    cleaning_summary: str

    # --- Phase 2: Cleaning Validation Loop ---
    post_cleaning_profile: dict
    validation_decision: dict
    post_remediation_profile: Optional[dict]
    cleaning_validation_iterations: int

    # --- Phase 3: Feature Engineering ---
    # feature_eng_decision: dict  # Stores FeatureEngDecision.model_dump()
    semantic_decisions: dict
    numeric_decisions: dict
    categorical_decisions: dict
    engineered_df_path: str     # Path to post-FE parquet
    feature_cols: List[str]     # Final assembler cols list
    feature_validation_report: dict # Structural checks (Level 1)
    feature_stats_report: dict  # Statistical profile (Level 2)
    validation_iterations: int
    validation_passed: bool
    validation_failures: List[str]

    # --- Phase 4: Model Training Loop ---
    model_decision: dict
    model_path: str
    predictions_path: str
    eval_decision: dict
    model_iterations: int

    # --- Final Output ---
    feature_importances: List[dict]
    insights_narrative: str

### define feature subagents


In [ ]:
import os

def load_skill(filename: str) -> str:
    """Read a skill.md file from disk (supports both SKILLS/ and skills/)."""
    # candidate_paths = [
    #     os.path.join("SKILLS", filename),
    #     os.path.join("skills", filename),
    # ]

    # for path in candidate_paths:
    #     if os.path.exists(path):
    #         with open(path, "r") as f:
    #             return f.read()
    path = os.path.join("SKILLS", filename)
    with open(path, "r") as f:
        return f.read()

    raise FileNotFoundError(f"Skill file not found in SKILLS/ or skills/: {filename}")

def semantic_agent_node(state: AgentState) -> AgentState:
    profile = state.get("post_remediation_profile") or state['post_cleaning_profile']
    skill_doc = load_skill("skill_semantic_boolean_expansion.md")

    system_prompt=f"""You are the semantic feature engineer for a PySpark Pipeline predicting US accident severity.
    Your only job is to look at compound text columns and extract boolean flags using regex

    ===SKILL MANUAL===
    {skill_doc}

    CRITICAL RULES:
    - EXHAUSTIVE COVERAGE: you MUST generate at least 8 to 10 distinct flags for Weather_Condition (eg. is_rain, is_snow, is_fog...)
    - You must generate at least 6 to 8 distinct flags for Street (eg. is_highway, is_local_street, is_country_road)
    - Write regex strictly in lowercase
"""
    user_prompt=f"Data profile:\n{profile}\n\nGenerate the semantic expansions."

    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        response_format=SemanticDecisionList
    )

    #ouput logs for tracking agent behaviour
    decision = response.choices[0].message.parsed
    print("\n" + "="*60)
    print("SEMANTIC AGENT DECISIONS")
    print("="*60)
    for exp in decision.semantic_expansions:
        print(f"{exp.source_column:<18} -> {exp.target_column:<20} (Regex: {exp.regex_pattern})")
        print(f"\nRationale: {exp.rationale}")
    print(f"\nOverall Rationale: {decision.overall_rationale}")
    print("="*60 + "\n")

    return {"semantic_decisions": decision.model_dump()}


def numeric_agent_node(state: AgentState) -> AgentState:
    profile = state.get("post_remediation_profile") or state["post_cleaning_profile"]
    skills = "\n---\n".join([
        load_skill("skill_extract_time_features.md"),
        load_skill("skill_bin_numeric.md"),
        load_skill("skill_compute_interaction_features.md"),
        load_skill("skill_scale_numeric.md")
    ])

    system_prompt = f"""You are the Numeric Feature Engineer for a PySpark pipeline predicting US accident severity.
Your job is to handle timestamps, numeric binning, numeric interactions, and continuous scaling.

=== SKILL MANUALS ===
{skills}

CRITICAL RULES:
- BINNING: You MUST apply 'skill_bin_numeric' to discretize at least 'Temperature(F)' and 'Visibility(mi)'. Use 'custom_cuts'.
- INTERACTIONS: You MUST invent at least ONE novel, logical interaction feature using multiply, add, or ratio.
- SCALING: Only True if Logistic Regression is the likely baseline model.
"""
    user_prompt = f"Data Profile:\n{profile}\n\nGenerate the numeric transformations."

    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        response_format=NumericDecisionList,
    )

    #ouput logs for tracking agent behaviour
    decision = response.choices[0].message.parsed
    print("\n" + "="*60)
    print("NUMERIC AGENT DECISIONS")
    print("="*60)
    if decision.time_features:
        print(f"Time Features: {decision.time_features.features_to_extract}")
        print(f"Rationale: {decision.time_features.rationale}")
    if decision.bins:
        for b in decision.bins:
            print(f"Binning: {b.column} ({b.strategy})")
            print(f"Rationale: {b.rationale}")
    if decision.interactions:
        for ix in decision.interactions:
            print(f"Interaction: {ix.col_a} {ix.op} {ix.col_b}")
            print(f"Rationale: {ix.rationale}")

    print(f"Scaling Applied: {decision.apply_scaling} (Strategy: {decision.scaling_strategy})")
    if decision.scaling_decisions:
        for s in decision.scaling_decisions:
            print(f" - {s.column}: {s.action} | {s.rationale}")
    if decision.scaling_rationale:
        print(f" Rationale: {decision.scaling_rationale}")

    print(f"\nOverall Rationale: {decision.overall_rationale}")
    print("="*60 + "\n")

    return {"numeric_decisions": decision.model_dump()}



def categorical_agent_node(state: AgentState) -> AgentState:
    profile = state.get("post_remediation_profile") or state["post_cleaning_profile"]
    skills = "\n---\n".join([
        load_skill("skill_encode_categorical.md"),
        load_skill("skill_drop_columns.md")
    ])

    system_prompt = f"""You are the Categorical Feature Engineer for a PySpark pipeline predicting US accident severity.
Your job is to encode the remaining string columns and designate which original columns to drop.

=== SKILL MANUALS ===
{skills}

CRITICAL RULES:
- For high cardinality columns (>20 unique values) like City, State, or Zipcode, you MUST use 'string_index_only' to prevent OOM errors.
- Always include 'Start_Time' and original encoded string columns in 'drop_after_encoding'.
"""
    user_prompt = f"Data Profile:\n{profile}\n\nGenerate the categorical encodings and drop list."

    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        response_format=CategoricalDecisionList,
    )

    decision = response.choices[0].message.parsed
    print("\n" + "="*60)
    print("CATEGORICAL AGENT DECISIONS")
    print("="*60)
    for enc in decision.column_encodings:
        print(f"Encode: {enc.column:<22} -> {enc.action}")
        print(f" | Rationale: {enc.rationale}")

    print(f"\nColumns to Drop: {decision.drop_after_encoding}")
    print(f"\nOverall Rationale: {decision.overall_rationale}")
    print("="*60 + "\n")

    return {"categorical_decisions": decision.model_dump()}

### execute feature engineering pyspark code (refactored for sub agent architecture)

In [ ]:
def execute_feature_eng(state: AgentState):
    from pyspark.sql import functions as F
    from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Bucketizer, StandardScaler, MinMaxScaler
    from pyspark.ml import Pipeline
    import numpy as np

    df = cleaned_df

    # sub-agent decisions
    sem = SemanticDecisionList.model_validate(state["semantic_decisions"])
    num = NumericDecisionList.model_validate(state["numeric_decisions"])
    cat = CategoricalDecisionList.model_validate(state["categorical_decisions"])

    assembler_cols = []

    # 1. time features
    if num.time_features:
        for feat in num.time_features.features_to_extract:
            if feat == "is_rush_hour":
                df = df.withColumn("is_rush_hour", F.when(F.hour("Start_Time").between(7,9) | F.hour("Start_Time").between(17,19), 1).otherwise(0))
                assembler_cols.append("is_rush_hour")

    # 2. semantic boolean expansions
    for exp in sem.semantic_expansions:
        if exp.source_column in df.columns:
            lower_col = F.lower(F.col(exp.source_column))
            df = df.withColumn(exp.target_column, lower_col.rlike(exp.regex_pattern).cast("integer")).fillna(0, subset=[exp.target_column])
            assembler_cols.append(exp.target_column)

    # 3. numeric interactions
    ops = {"multiply": lambda a, b: F.col(a) * F.col(b),
           "add": lambda a, b: F.col(a) + F.col(b),
           "ratio": lambda a, b: F.col(a) / (F.col(b) + F.lit(1e-6))
    }
    for ix in num.interactions:
        if ix.col_a in df.columns and ix.col_b in df.columns:
            out_col = f"{ix.col_a}_{ix.op}_{ix.col_b}"
            df = df.withColumn(out_col, ops[ix.op](ix.col_a, ix.col_b))
            assembler_cols.append(out_col)

    # 4. numeric binning
    for b in num.bins:
        if b.column in df.columns:
            if b.strategy == "equal_width":
                mn = df.agg({b.column: "min"}).collect()[0][0]
                mx = df.agg({b.column: "max"}).collect()[0][0]
                splits = [float(x) for x in np.linspace(mn, mx, b.n_bins + 1)]
                splits[0], splits[-1] = float("-inf"), float("inf")
            else:
                splits = [float("-inf")] + [float(c) for c in b.cuts] + [float("inf")]

            out_col = f"{b.column}_bin"
            df = Bucketizer(splits=splits, inputCol=b.column, outputCol=out_col, handleInvalid="keep").transform(df)
            assembler_cols.append(out_col)

            bin_descriptions = []
            for i in range(len(splits) -1):
                lower = "MIN" if splits[i] == float('-inf') else round(splits[i],2)
                upper = "MAX" if splits[i+1] == float('inf') else round(splits[i+1],2)
                bin_descriptions.append(f"Bin {i} [{lower} to {upper}]")

            mapping_str = " | ".join(bin_descriptions)
            print(f"Binned: {b.column} -> {b.column}_bin")
            print(f" -> {mapping_str}")

    # 5. categorical encoding
    indexers, encoders = [], []
    for enc in cat.column_encodings:
        if enc.column in df.columns:
            if enc.action == "string_index_ohe":
                indexers.append(StringIndexer(inputCol=enc.column, outputCol=f"{enc.column}_idx", handleInvalid="keep"))
                encoders.append(OneHotEncoder(inputCol=f"{enc.column}_idx", outputCol=f"{enc.column}_ohe"))
                assembler_cols.append(f"{enc.column}_ohe")
            elif enc.action == "string_index_only":
                indexers.append(StringIndexer(inputCol=enc.column, outputCol=f"{enc.column}_idx", handleInvalid="keep"))
                assembler_cols.append(f"{enc.column}_idx")
            elif enc.action == "passthrough":
                assembler_cols.append(enc.column)

    if indexers or encoders:
        df = Pipeline(stages=indexers + encoders).fit(df).transform(df)

    # 6. numeric scaling
    if num.apply_scaling and num.scaling_strategy:
        cols_to_scale = [s.column for s in num.scaling_decisions if s.action == 'scale' and s.column in df.columns]
        for col in cols_to_scale:
            if col not in assembler_cols:
                assembler_cols.append(col) # Ensure original scaled numeric cols are assembled
            vec_col, scaled_col = f"{col}_vec", f"{col}_scaled"
            asm = VectorAssembler(inputCols=[col], outputCol=vec_col)
            scaler = StandardScaler(inputCol=vec_col, outputCol=scaled_col, withMean=True, withStd=True) if num.scaling_strategy == "standard" else MinMaxScaler(inputCol=vec_col, outputCol=scaled_col)
            df = Pipeline(stages=[asm, scaler]).fit(df).transform(df)
            df = df.drop(vec_col, col).withColumnRenamed(scaled_col, col)

    # Ensure un-scaled numerics intended for the model are caught if not handled above
    # (Optional logic depending on how strict your scaling_decisions list is)

    # 7. Assemble Vector
    # Deduplicate in case Python appended safely twice
    assembler_cols = list(dict.fromkeys([c for c in assembler_cols if c != "Severity" and c != "Severity_Binary"]))

    if assembler_cols:
        assembler = VectorAssembler(inputCols=assembler_cols, outputCol="features", handleInvalid="skip")
        df = assembler.transform(df)

    # 8. Save
    out_path = "data/engineered_df.parquet"
    df.write.mode("overwrite").parquet(out_path)
    print(f"Feature engineering compile and executed. Assembled {len(assembler_cols)} features.")

    return {
        "feature_cols": assembler_cols,
        "engineered_df_path": out_path
    }

## validation checks for feature engineering.

In [ ]:
def validate_feature_eng(state: AgentState, df) -> dict:
    from pyspark.sql import functions as F

    results = {"passed": True, "checks": []}

    def check(name, condition, detail):
        status = "PASS" if condition else "FAIL"
        results["checks"].append({"check": name, "status": status, "detail": detail})
        if not condition:
            results["passed"] = False

    feature_cols = state.get("feature_cols", [])

    check(
        "no_target_leakage",
        "Severity" not in feature_cols,
        "Severity excluded from assembler_cols" if "Severity" not in feature_cols else "Severity present in assembler_cols"
    )

    for col in feature_cols:
        if col in df.columns:
            n_null = df.filter(F.col(col).isNull()).count()
            check(f"no_nulls_{col}", n_null == 0, f"{n_null} nulls found" if n_null > 0 else "Clean")

    # cehck time features are in expected range
    num_decisions = state.get("numeric_decisions", {})
    time_feats = num_decisions.get("time_features", {})
    if time_feats and "features_to_extract" in time_feats:
        for feat in time_feats["features_to_extract"]:
            if feat == "is_rush_hour" and feat in df.columns:
                out_of_range = df.filter((F.col(feat) < 0) | (F.col(feat) > 1)).count()
                check(f"range_{feat}", out_of_range == 0, f"{out_of_range} rows out of [0, 1]" if out_of_range else "All in [0, 1]")

    # check features vector exists
    check("features_vector_exists", "features" in df.columns, "VectorAssembler output present" if "features" in df.columns else "features col missing")

    # check dropped columns are actually gone
    # still_present = [c for c in decision.drop_after_encoding if c in df.columns]
    # check("dropped_cols_gone", len(still_present) == 0, f"Still in df: {still_present}" if still_present else "All dropped")

    print("\n=== Feature Engineering Validation Report ===")
    for c in results["checks"]:
        symbol = "✓" if c["status"] == "PASS" else "x"
        print(f"[{symbol}] {c['check']}: {c['detail']}")
    print(f"\nOverall: {'PASSED' if results['passed'] else 'FAILED'}")

    failures = [
        f"{c['check']}: {c['detail']}"
        for c in results["checks"]
        if c["status"] == "FAIL"
    ]

    return {
        "feature_validation_report": results,
        "validation_passed": results["passed"],
        "validation_failures": failures,
    }

In [ ]:
def profile_engineered_features(df, state: AgentState) -> dict:
    from pyspark.sql import functions as F
    from pyspark.ml.functions import vector_to_array

    report = {}

    # 1. Class balance
    target_col = "Severity_Binary" if "Severity_Binary" in df.columns else "Severity"
    class_dist = df.groupBy(target_col).count().orderBy(target_col).collect()
    total = df.count()
    report["class_balance"] = {
        r[target_col]: {
            "count": r["count"],
            "pct": round(r["count"] / total * 100, 1)
        } for r in class_dist
    }

    # Class weight ratio for the Model Selection agent
    if 0 in report["class_balance"] and 1 in report["class_balance"]:
        n_majority = report["class_balance"][0]["count"]
        n_minority = report["class_balance"][1]["count"]
        report["class_weight_ratio"] = round(n_majority / n_minority, 2)

    # 2. Time feature distributions
    num_decisions = state.get("numeric_decisions", {})
    time_feats = num_decisions.get("time_features", {})
    if time_feats and "features_to_extract" in time_feats:
        for feat in time_feats["features_to_extract"]:
            if feat in df.columns:
                dist = df.groupBy(feat).count().orderBy(feat).collect()
                report[f"{feat}_distribution"] = {r[feat]: r["count"] for r in dist}

    # 3. Cardinality of index-encoded columns
    idx_cols = [c for c in df.columns if c.endswith("_idx")]
    for col in idx_cols:
        report[f"{col}_cardinality"] = df.select(col).distinct().count()

    # 5. Feature vector summary
    if "features" in df.columns:
        feat_array = df.select(vector_to_array("features").alias("f"))
        sizes = feat_array.select(F.size("f").alias("size")).distinct().collect()
        report["feature_vector_sizes"] = [r["size"] for r in sizes] # Should be exactly 1

    report["expected_feature_count"] = len(state.get("feature_cols", []))

    return {"feature_stats_report": report}

## langgraph

In [ ]:
MAX_VALIDATION_RETRIES = 2

def validate_feature_eng_node(state: AgentState):
    df = spark.read.parquet(state["engineered_df_path"])

    val_report = validate_feature_eng(state, df)
    updated_state = {**state, **val_report}

    if updated_state["validation_passed"]:
        stats_report = profile_engineered_features(df, updated_state)
        updated_state.update(stats_report)
    else:
        updated_state["validation_iterations"] = state.get("validation_iterations", 0) + 1

    return updated_state


def route_after_validation(state: AgentState):
    if state.get("validation_passed", False):
        return "end"
    if state.get("validation_iterations", 0) <= MAX_VALIDATION_RETRIES:
        return "retry"
    return "end"


# subgraph for now will integrate to full workflow
def build_feature_eng_subgraph():
    workflow = StateGraph(AgentState)

    #sub agent nodes
    workflow.add_node("semantic_agent", semantic_agent_node)
    workflow.add_node("numeric_agent", numeric_agent_node)
    workflow.add_node("categorical_agent", categorical_agent_node)
    workflow.add_node("compiler", execute_feature_eng)
    workflow.add_node("validate", validate_feature_eng_node)

    # define edges
    workflow.set_entry_point("semantic_agent")
    workflow.add_edge("semantic_agent", "numeric_agent")
    workflow.add_edge("numeric_agent", "categorical_agent")
    workflow.add_edge("categorical_agent", "compiler")
    workflow.add_edge("compiler", "validate")
    workflow.add_conditional_edges(
        "validate",
        route_after_validation,
        {
            "retry": "semantic_agent",
            "end": END,
        },
    )

    return workflow.compile()

feat_eng = build_feature_eng_subgraph()

In [ ]:
# initial_state = {
#     "cleaning_summary": "Dropped pre-2019 rows, collapsed Wind_Direction to 5 cats, boolean-expanded Weather_Condition.",
#     "post_cleaning_profile": {
#         "num_rows": 1000, # Use a small number for testing
#         "columns": ["Start_Time", "State", "Temperature(F)", "Severity"],
#         # ... (Add the rest of the profile dict here from the architecture doc) ...
#         "start_time_sample": ["2019-02-08 05:46:00", "2019-02-08 06:07:59"]
#     }
# }

initial_state = {
    "cleaning_summary": "Manually cleaned data: handled nulls. Mapped Severity to Severity_Binary (Classes 1 & 2 -> 0; Classes 3 & 4 -> 1) for binary classification.",
    "post_cleaning_profile": {
        "num_rows": 5270673,
        "num_cols": 50,
        "columns": [
            "Airport_Code", "Start_Time_Month", "Severity_Binary", "Start_Time", "End_Time",
            "Start_Lat", "Start_Lng", "Distance(mi)", "Street", "City", "County", "State",
            "Zipcode", "Timezone", "Temperature(F)", "Humidity(%)", "Pressure(in)",
            "Visibility(mi)", "Wind_Direction", "Wind_Speed(mph)", "Precipitation(in)",
            "Weather_Condition", "Amenity", "Bump", "Crossing", "Give_Way", "Junction",
            "No_Exit", "Railway", "Roundabout", "Station", "Stop", "Traffic_Calming",
            "Traffic_Signal", "Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight",
            "Astronomical_Twilight", "Start_Time_Year", "Start_Time_Day_of_Week",
            "Start_Time_Hour", "Start_Time_is_Weekend", "End_Time_Year", "End_Time_Month",
            "End_Time_Day_of_Week", "End_Time_Hour", "End_Time_is_Weekend",
            "Duration_Seconds", "Duration_Minutes", "Duration_Hours"
        ],
        "schema": {
            "Start_Time": "timestamp",
            "End_Time": "timestamp",
            "Severity_Binary": "int",
            "Temperature(F)": "double",
            "Humidity(%)": "double",
            "Pressure(in)": "double",
            "Visibility(mi)": "double",
            "Wind_Speed(mph)": "double",
            "Precipitation(in)": "double",
            "Distance(mi)": "double",
            "Duration_Minutes": "double",
            "Weather_Condition": "string",
            "City": "string",
            "State": "string",
            "Zipcode": "string",
            "Start_Time_Hour": "int",
            "Start_Time_is_Weekend": "int"
        },
        "null_counts": {},
        "categorical_columns": {
            "Weather_Condition": {
                "n_unique": 127,
                "sample_values": ["Fair", "Cloudy", "Mostly Cloudy", "Partly Cloudy", "Light Rain"]
            },
            "Wind_Direction": {
                "n_unique": 10,
                "sample_values": ["CALM", "S", "W", "N", "E"]
            },
            "City": {
                "n_unique": 11289,
                "sample_values": ["Miami", "Los Angeles", "Houston", "Charlotte", "Orlando"]
            },
            "State": {
                "n_unique": 49,
                "sample_values": ["CA", "FL", "TX", "SC", "NY"]
            },
            "Zipcode": {
                "n_unique": 636412,
                "sample_values": ["91761", "33186", "92407", "91706", "92507"]
            },
            "Street": {
                "n_unique": 281073,
                "sample_values": ["I-95 S", "I-95 N", "I-5 N", "I-5 S", "I-10 W"]
            },
            "Sunrise_Sunset": {
                "n_unique": 2,
                "sample_values": ["Day", "Night"]
            }
        },
        "numeric_cols": [
            "Temperature(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)",
            "Wind_Speed(mph)", "Precipitation(in)", "Distance(mi)", "Duration_Minutes",
            "Start_Time_Hour", "Start_Time_Month"
        ],
        "numeric_summary": {
            "min": {"Temperature(F)": "-89.0", "Humidity(%)": "1.0", "Pressure(in)": "0.0", "Visibility(mi)": "0.0", "Wind_Speed(mph)": "0.0", "Precipitation(in)": "0.0", "Distance(mi)": "0.0", "Duration_Minutes": "5.2167"},
            "max": {"Temperature(F)": "207.0", "Humidity(%)": "100.0", "Pressure(in)": "58.63", "Visibility(mi)": "140.0", "Wind_Speed(mph)": "1087.0", "Precipitation(in)": "36.47", "Distance(mi)": "99.952", "Duration_Minutes": "17279.9833"},
            "mean": {"Temperature(F)": "61.2189", "Humidity(%)": "64.6345", "Pressure(in)": "29.3772", "Visibility(mi)": "9.0612", "Wind_Speed(mph)": "7.3783", "Precipitation(in)": "0.0058", "Distance(mi)": "0.6439", "Duration_Minutes": "120.2057"},
            "stddev": {"Temperature(F)": "19.2101", "Humidity(%)": "22.8063", "Pressure(in)": "1.0959", "Visibility(mi)": "2.6049", "Wind_Speed(mph)": "5.4732", "Precipitation(in)": "0.0517", "Distance(mi)": "1.7643", "Duration_Minutes": "240.769"}
        },
        "boolean_cols": [
            "Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit",
            "Railway", "Roundabout", "Station", "Stop", "Traffic_Calming", "Traffic_Signal"
        ],
        "class_distribution": {
            0: {"count": 4531863, "pct": 85.98},
            1: {"count": 738810, "pct": 14.02}
        },
        "class_weight_ratio": 6.13,
        "start_time_sample": ["2019-01-01 00:00:00"]
    }
}


print("Starting Feature Engineering Phase...\n")
final_state = feat_eng.invoke(initial_state)

print("\n--- Pipeline Complete! ---")
print(f"Final features passed to VectorAssembler: {final_state['feature_cols']}")

Starting Feature Engineering Phase...


SEMANTIC AGENT DECISIONS
Weather_Condition  -> is_rain              (Regex: rain|storm|drizzle|shower|thunder)
Rationale: Captures conditions indicating rain or related precipitation that can affect driving conditions.
Weather_Condition  -> is_snow              (Regex: snow|sleet|ice|pellets|squall|blizzard)
Rationale: Captures winter weather conditions including various forms of snow and ice presence.
Weather_Condition  -> is_fog               (Regex: fog|haze|mist|smoke|low visibility)
Rationale: Captures conditions that involve reduced visibility due to fog or similar conditions.
Weather_Condition  -> is_windy             (Regex: windy|gale|blustery|breezy)
Rationale: Captures conditions where wind could impact driving.
Weather_Condition  -> is_cloudy            (Regex: cloudy|overcast|partly cloudy|mostly cloudy)
Rationale: Indicates general cloud coverage which may affect visibility and driving conditions.
Weather_Condition  -> is_clear     

Feature engineering compile and executed. Assembled 35 features.



=== Feature Engineering Validation Report ===
[✓] no_target_leakage: Severity excluded from assembler_cols
[✓] no_nulls_is_rush_hour: Clean
[✓] no_nulls_is_rain: Clean
[✓] no_nulls_is_snow: Clean
[✓] no_nulls_is_fog: Clean
[✓] no_nulls_is_windy: Clean
[✓] no_nulls_is_cloudy: Clean
[✓] no_nulls_is_clear: Clean
[✓] no_nulls_is_storm: Clean
[✓] no_nulls_is_precipitation: Clean
[✓] no_nulls_is_highway: Clean
[✓] no_nulls_is_local_street: Clean
[✓] no_nulls_is_county_road: Clean
[✓] no_nulls_is_state_road: Clean
[✓] no_nulls_is_at_intersection: Clean
[✓] no_nulls_is_one_way: Clean
[✓] no_nulls_is_paved_road: Clean
[✓] no_nulls_Distance(mi)_ratio_Duration_Minutes: Clean
[✓] no_nulls_Temperature(F)_multiply_Humidity(%): Clean
[✓] no_nulls_Visibility(mi)_bin: Clean
[✓] no_nulls_Temperature(F)_bin: Clean
[✓] no_nulls_Weather_Condition_idx: Clean
[✓] no_nulls_Wind_Direction_ohe: Clean
[✓] no_nulls_City_idx: Clean
[✓] no_nulls_State_idx: Clean
[✓] no_nulls_Zipcode_idx: Clean
[✓] no_nulls_Street_


--- Pipeline Complete! ---
Final features passed to VectorAssembler: ['is_rush_hour', 'is_rain', 'is_snow', 'is_fog', 'is_windy', 'is_cloudy', 'is_clear', 'is_storm', 'is_precipitation', 'is_highway', 'is_local_street', 'is_county_road', 'is_state_road', 'is_at_intersection', 'is_one_way', 'is_paved_road', 'Distance(mi)_ratio_Duration_Minutes', 'Temperature(F)_multiply_Humidity(%)', 'Visibility(mi)_bin', 'Temperature(F)_bin', 'Weather_Condition_idx', 'Wind_Direction_ohe', 'City_idx', 'State_idx', 'Zipcode_idx', 'Street_idx', 'Sunrise_Sunset_ohe', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)', 'Distance(mi)', 'Duration_Minutes']


In [ ]:
FEATURE_ENGINEERED_PARQUET_DIR = PROJECT_ROOT / "data" / "engineered_df.parquet"

if not FEATURE_ENGINEERED_PARQUET_DIR.exists():
    raise FileNotFoundError(
        f"Feature engineered folder not found at: {FEATURE_ENGINEERED_PARQUET_DIR}\n"
        "Run feature engineering first, or update FEATURE_ENGINEERED_PARQUET_DIR."
    )

print("Feature engineering parquet dir:", FEATURE_ENGINEERED_PARQUET_DIR)
feature_df = spark.read.parquet(str(FEATURE_ENGINEERED_PARQUET_DIR))

feature_df.show(5)

Feature engineering parquet dir: /Users/bryan/Documents/Y2S2/BT4221/Tutorials/work/data/engineered_df.parquet


+------------+----------------+--------+-------------------+-------------------+---------+-------------------+---------------+--------+--------+-----+-------+-----------+--------------+--------------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+--------------+--------------+-----------------+---------------------+---------------+----------------------+---------------+---------------------+-------------+--------------+--------------------+-------------+-------------------+----------------+-------------------+------------+-------+-------+------+-------+--------+---------------+-------+--------+----------+---------------+---------------+---------------+----------+---------------+---------------+-----------------------------------+------------------+------------------+---------------------+--------+---------+-----------+----------+------------------+------------------+------------------+------------------+-----------

In [ ]:
feature_df.columns

['Airport_Code',
 'Start_Time_Month',
 'Severity',
 'Start_Time',
 'End_Time',
 'Start_Lat',
 'Start_Lng',
 'Street',
 'City',
 'County',
 'State',
 'Zipcode',
 'Timezone',
 'Wind_Direction',
 'Weather_Condition',
 'Amenity',
 'Bump',
 'Crossing',
 'Give_Way',
 'Junction',
 'No_Exit',
 'Railway',
 'Roundabout',
 'Station',
 'Stop',
 'Traffic_Calming',
 'Traffic_Signal',
 'Sunrise_Sunset',
 'Civil_Twilight',
 'Nautical_Twilight',
 'Astronomical_Twilight',
 'Start_Time_Year',
 'Start_Time_Day_of_Week',
 'Start_Time_Hour',
 'Start_Time_is_Weekend',
 'End_Time_Year',
 'End_Time_Month',
 'End_Time_Day_of_Week',
 'End_Time_Hour',
 'End_Time_is_Weekend',
 'Duration_Seconds',
 'Duration_Hours',
 'is_rush_hour',
 'is_rain',
 'is_snow',
 'is_fog',
 'is_fair',
 'is_windy',
 'is_thunderstorm',
 'is_hail',
 'is_sunny',
 'is_highway',
 'is_local_street',
 'is_country_road',
 'is_urban_street',
 'is_one_way',
 'is_traffic_sign',
 'is_intersection',
 'Temperature(F)_multiply_Humidity(%)',
 'Visibility